<style>
  @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;600;700&display=swap');
</style>

<div style="font-family:'Montserrat', ui-sans-serif, system-ui, -apple-system, 'Segoe UI', Roboto, Helvetica, Arial; padding:10px 0 6px 0; border-bottom:1px solid #ef4444;">
  <div style="display:flex; align-items:flex-start; gap:12px;">
    <img src="../assets/aiphet-logo.png" alt="Aiphet logo" style="width:44px;height:44px;border-radius:10px;object-fit:contain; margin-top:2px;" />
    <div style="line-height:1.15;">
      <div style="font-size:26px;font-weight:700;letter-spacing:0.2px;">Aiphet</div>
      <div style="font-size:14px;color:#4b5563;margin-top:2px;">Fine-tuning (LoRA/QLoRA) of a public Qwen model for high-quality PROMs, PREMs, and general medical forms</div>
      <div style="font-size:12px;color:#6b7280;margin-top:6px;">Author: Pablo Pimàs</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">Email: pablo@pimas.cat</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">Date: February 22, 2026</div>
      <div style="font-size:12px;color:#6b7280;margin-top:2px;">License: CC BY 4.0 (SPDX: CC-BY-4.0)</div>
      <div style="font-size:12px;color:#6b7280; font-weight:600; margin-top:2px;">#AIP-173</div>
    </div>
  </div>
</div>


## Table of Contents

1. Fine-Tuning Qwen2.5-7B-Instruct-4bit for FHIR R4 QuestionnaireItem Generation
2. Reproducibility Protocol
3. Dataset Statement and Governance
4. Data Loading and ChatML Parsing
5. FHIR Structural Audit
6. Training Configuration and Rationale
7. Reproducible Train/Validation Split
8. Data Materialization for Training
9. Training Execution (MLX-LM)
10. Quantitative Evaluation
11. Qualitative Analysis
12. Limitations, Ethics, and Threats to Validity
13. References
14. How to Cite

# Fine-Tuning Qwen2.5-7B-Instruct-4bit for FHIR R4 QuestionnaireItem Generation

## Scope

This notebook documents an end-to-end, reproducible training workflow for adapting **mlx-community/Qwen2.5-7B-Instruct-4bit** to generate **FHIR R4 QuestionnaireItem-like JSON** from ChatML-formatted prompts in a clinical PROMs/PREMs context.

The workflow is designed for:

- Apple Silicon execution using MLX/MLX-LM
- parameter-efficient adaptation (LoRA/QLoRA-style setup)
- structured-output generation under FHIR-oriented constraints

## Main Contributions

1. Reproducible data preparation and validation pipeline for JSONL + ChatML records.
2. Structured fine-tuning protocol for FHIR Questionnaire item generation.
3. Quantitative evaluation of structural validity and schema-conformance behavior.
4. Qualitative error analysis of generated medical questionnaire items.
5. Explicit legal/licensing caveats for standardized instrument content.

## Research Context

Patient-reported outcomes (PROMs) and patient-reported experience measures (PREMs) often require strict structural interoperability.  
This notebook focuses on generating machine-usable questionnaire items compatible with **HL7 FHIR R4 Questionnaire semantics**.

## References (Core Concepts)

- HL7 FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- LoRA (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- QLoRA (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- Qwen documentation: https://qwen.readthedocs.io/
- MLX: https://github.com/ml-explore/mlx
- MLX-LM: https://github.com/ml-explore/mlx-lm

## Compliance Note

This notebook is provided for research and educational use.  
If standardized questionnaire text is redistributed or used commercially, licensing and copyright obligations must be verified instrument-by-instrument.

## Reproducibility Protocol

To support repeatability, this notebook follows a deterministic workflow where possible.

### Execution Environment

- Platform target: Apple Silicon (macOS) with MLX/MLX-LM
- Python environment managed with pinned package versions from `requirements.txt`
- Notebook and scripts executed from repository root to preserve relative paths

### Determinism Settings

We will enforce and report:

- global random seed
- dataset split seed
- explicit train/validation partition method
- fixed configuration values for model and optimization

> Note: exact bitwise reproducibility is not always guaranteed across hardware/software backends, but seed and environment control substantially reduce variance.

### Experimental Tracking

For each training run, we will record:

1. model identifier
2. dataset file path and row counts
3. preprocessing/validation checks
4. hyperparameters (learning rate, LoRA rank, batch size, sequence length, steps)
5. runtime metadata (device, library versions)
6. evaluation outputs and qualitative examples

### Run Order

This notebook should be executed top-to-bottom in a single pass:

1. environment checks
2. dataset loading and validation
3. split generation
4. training
5. evaluation and error analysis

### References

- ACM Artifact Review and Badging: https://www.acm.org/publications/policies/artifact-review-and-badging-current
- The Turing Way (Reproducibility): https://the-turing-way.netlify.app/reproducible-research/reproducible-research
- MLX: https://github.com/ml-explore/mlx

In [1]:
# Reproducibility: environment and package snapshot

from __future__ import annotations

import importlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

# --- Fixed seeds for reproducibility ---
GLOBAL_SEED = 173
random.seed(GLOBAL_SEED)
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

# --- Helper to get package versions safely ---
def safe_version(module_name: str) -> str:
    """Return the installed version for a module, or a fallback label.

    Parameters
    ----------
    module_name:
        Importable module name (e.g., "mlx", "datasets").

    Returns
    -------
    str
        Module version string when available, otherwise "unknown" or
        "not-installed" if import fails.
    """
    try:
        module = importlib.import_module(module_name)
        return getattr(module, "__version__", "unknown")
    except Exception:
        return "not-installed"


# --- Detect repository root heuristically ---
def find_repo_root(start: Path) -> Path:
    """Find repository root by locating folders that match this project layout.

    The function walks from the current working directory up to parent folders
    and returns the first directory containing both `README.md` and `data/`.

    Parameters
    ----------
    start:
        Initial path used as search origin.

    Returns
    -------
    Path
        Detected repository root; if none is found, returns `start.resolve()`.
    """
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return start.resolve()


repo_root = find_repo_root(Path.cwd())
dataset_path = repo_root / "data" / "aiprom-dataset-fhir-4-150.jsonl"

# --- System/runtime info ---
runtime_info = {
    "timestamp_utc": datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cwd": str(Path.cwd().resolve()),
    "repo_root": str(repo_root),
    "dataset_exists": dataset_path.exists(),
    "dataset_path": str(dataset_path),
    "seed": GLOBAL_SEED,
    "packages": {
        "mlx": safe_version("mlx"),
        "mlx_lm": safe_version("mlx_lm"),
        "datasets": safe_version("datasets"),
        "yaml": safe_version("yaml"),
        "numpy": safe_version("numpy"),
        "pandas": safe_version("pandas"),
        "wandb": safe_version("wandb"),
    },
}

# --- Optional: git metadata if available ---
if shutil.which("git"):
    try:
        commit = subprocess.check_output(
            ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
            text=True
        ).strip()
    except Exception:
        commit = "unavailable"
else:
    commit = "git-not-found"

runtime_info["git_commit"] = commit

print(json.dumps(runtime_info, indent=2))

/Users/CAE9/aiprom-llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "timestamp_utc": "2026-02-22T16:24:53Z",
  "python_version": "3.14.3",
  "platform": "macOS-26.3-arm64-arm-64bit-Mach-O",
  "machine": "arm64",
  "processor": "arm",
  "cwd": "/Users/CAE9/aiprom-llm/lab",
  "repo_root": "/Users/CAE9/aiprom-llm",
  "dataset_exists": true,
  "dataset_path": "/Users/CAE9/aiprom-llm/data/aiprom-dataset-fhir-4-150.jsonl",
  "seed": 173,
  "packages": {
    "mlx": "unknown",
    "mlx_lm": "0.30.7",
    "datasets": "4.5.0",
    "yaml": "6.0.3",
    "numpy": "2.4.2",
    "pandas": "3.0.1",
    "wandb": "0.25.0"
  },
  "git_commit": "a29e4c4648e7d3ad2aeeef91d1f59c8bd92fbee0"
}


## Dataset Statement and Governance

### Primary Dataset

This study uses the JSONL dataset:

- `data/aiprom-dataset-fhir-4-150.jsonl`

Each row contains a ChatML transcript in a single `text` field, with a structured assistant target representing a **FHIR R4 QuestionnaireItem-like JSON fragment**.

### Structural Coverage

The dataset includes all 13 Questionnaire item types used in this project:

- `group`, `display`, `boolean`, `decimal`, `integer`, `date`, `dateTime`, `time`, `string`, `text`, `url`, `choice`, `open-choice`

### Clinical Content Scope

Prompt/content coverage includes validated PROMs-related instruments and clinical scales (e.g., EORTC QLQ-C30, PHQ-9, GAD-7, EQ-5D-5L, PROMIS, WPAI, HADS), plus symptom-oriented constructs such as pain, fatigue, anxiety, depression, and quality of life.

### Data Format Contract

The training objective is conditioned generation:

- **Input**: system + user turns in ChatML
- **Output**: assistant JSON with FHIR-oriented fields (`linkId`, `text`, `type`, `required`, `code`, and `answerOption` when applicable)

### Licensing and Usage Caveat

This notebook is for research/educational purposes.  
Some questionnaire content may correspond to third-party instruments with distinct copyright or licensing terms.

Before redistribution or commercial use, verify rights instrument-by-instrument.

See repository documentation for details:

- `README.md` (licensing overview and practical implications)
- `docs/datasets.md` (dataset scope and legal notes)

### References

- FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- FAIR principles for scientific data: https://www.go-fair.org/fair-principles/
- PROMIS (HealthMeasures): https://www.healthmeasures.net/explore-measurement-systems/promis

In [2]:
# Data loading: JSONL parsing + ChatML assistant JSON extraction

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any


ASSISTANT_MARKER = "<|im_start|>assistant\n"
END_MARKER = "<|im_end|>"


@dataclass
class RecordParseResult:
    row_id: int
    raw_record: dict[str, Any]
    text: str
    assistant_raw: str
    assistant_obj: dict[str, Any]


def extract_assistant_json_from_chatml(text: str) -> str:
    """Extract assistant JSON block from a ChatML transcript string.

    Parameters
    ----------
    text:
        Full ChatML transcript stored in the JSONL `text` field.

    Returns
    -------
    str
        Raw JSON text emitted by the assistant turn.

    Raises
    ------
    ValueError
        If assistant markers are missing or the extracted assistant block is empty.
    """
    start = text.rfind(ASSISTANT_MARKER)
    if start < 0:
        raise ValueError("Missing assistant marker in ChatML text")

    json_start = start + len(ASSISTANT_MARKER)
    json_end = text.find(END_MARKER, json_start)
    if json_end < 0:
        raise ValueError("Missing end marker after assistant block")

    assistant_raw = text[json_start:json_end].strip()
    if not assistant_raw:
        raise ValueError("Assistant block is empty")
    return assistant_raw


def load_fhir_jsonl_records(path: Path) -> tuple[list[RecordParseResult], list[str]]:
    """Load and parse JSONL rows into structured records for FHIR training.

    Each row is validated as:
    1) valid JSON object,
    2) containing a string `text` field,
    3) containing parseable assistant JSON in ChatML format.

    Parameters
    ----------
    path:
        Path to the dataset JSONL file.

    Returns
    -------
    tuple[list[RecordParseResult], list[str]]
        Parsed records and accumulated human-readable parsing errors.
    """
    results: list[RecordParseResult] = []
    errors: list[str] = []

    with path.open("r", encoding="utf-8") as f:
        for row_id, line in enumerate(f, start=1):
            if not line.strip():
                errors.append(f"Line {row_id}: empty line")
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                errors.append(f"Line {row_id}: invalid JSONL record ({exc})")
                continue

            if not isinstance(record, dict):
                errors.append(f"Line {row_id}: record is not a JSON object")
                continue

            text = record.get("text")
            if not isinstance(text, str):
                errors.append(f"Line {row_id}: missing or invalid 'text' field")
                continue

            try:
                assistant_raw = extract_assistant_json_from_chatml(text)
            except ValueError as exc:
                errors.append(f"Line {row_id}: {exc}")
                continue

            try:
                assistant_obj = json.loads(assistant_raw)
            except json.JSONDecodeError as exc:
                errors.append(f"Line {row_id}: assistant JSON decode error ({exc})")
                continue

            if not isinstance(assistant_obj, dict):
                errors.append(f"Line {row_id}: assistant payload is not a JSON object")
                continue

            results.append(
                RecordParseResult(
                    row_id=row_id,
                    raw_record=record,
                    text=text,
                    assistant_raw=assistant_raw,
                    assistant_obj=assistant_obj,
                )
            )

    return results, errors


dataset_path = Path(runtime_info["dataset_path"])
parsed_records, parse_errors = load_fhir_jsonl_records(dataset_path)

print(f"Dataset path: {dataset_path}")
print(f"Parsed records: {len(parsed_records)}")
print(f"Parse errors: {len(parse_errors)}")

if parse_errors:
    print("\nSample parse errors (up to 10):")
    for err in parse_errors[:10]:
        print(f"- {err}")

# show a compact preview of first parsed assistant object
if parsed_records:
    sample = parsed_records[0].assistant_obj
    print("\nFirst assistant object keys:", sorted(sample.keys()))

Dataset path: /Users/CAE9/aiprom-llm/data/aiprom-dataset-fhir-4-150.jsonl
Parsed records: 150
Parse errors: 0

First assistant object keys: ['code', 'linkId', 'text', 'type']


In [3]:
# FHIR structural audit: schema-oriented checks and coverage summary

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from typing import Any


FHIR_ALLOWED_TYPES = {
    "group",
    "display",
    "boolean",
    "decimal",
    "integer",
    "date",
    "dateTime",
    "time",
    "string",
    "text",
    "url",
    "choice",
    "open-choice",
}


@dataclass
class AuditSummary:
    total_records: int
    valid_records: int
    invalid_records: int
    pass_rate: float
    type_counts: dict[str, int]
    issue_counts: dict[str, int]


def is_valid_code_array(value: Any) -> bool:
    """Check whether `code` follows a minimal valid Coding[] shape.

    A valid `code` is a non-empty list of objects, each containing non-empty
    string fields `system` and `code`.

    Parameters
    ----------
    value:
        Arbitrary JSON value from assistant payload.

    Returns
    -------
    bool
        True if value matches the expected minimal structure, else False.
    """
    if not isinstance(value, list) or len(value) == 0:
        return False

    for coding in value:
        if not isinstance(coding, dict):
            return False
        system = coding.get("system")
        code_value = coding.get("code")
        if not isinstance(system, str) or not system.strip():
            return False
        if not isinstance(code_value, str) or not code_value.strip():
            return False

    return True


def has_valid_answer_option(payload: dict[str, Any]) -> bool:
    """Validate `answerOption` for choice/open-choice items.

    Minimal requirement used in this notebook:
    - `answerOption` exists and is a non-empty list of objects
    - each option contains at least one key starting with `value`

    Parameters
    ----------
    payload:
        Assistant JSON object for one training record.

    Returns
    -------
    bool
        True if answerOption passes this structural check.
    """
    answer_option = payload.get("answerOption")
    if not isinstance(answer_option, list) or len(answer_option) == 0:
        return False

    for option in answer_option:
        if not isinstance(option, dict):
            return False
        if not any(str(key).startswith("value") for key in option.keys()):
            return False

    return True


def audit_fhir_record(payload: dict[str, Any]) -> list[str]:
    """Run structural FHIR-oriented validation checks for one payload.

    Rules used:
    - `type` must exist and be one of FHIR_ALLOWED_TYPES
    - `linkId` must be a non-empty string
    - `required` (if present) must be boolean
    - `code` required for all non-group/non-display types
    - `text` required for all non-group/non-display types
    - `answerOption` required for `choice` and `open-choice`

    Parameters
    ----------
    payload:
        Assistant JSON object for one dataset row.

    Returns
    -------
    list[str]
        List of issue tags. Empty list means the record passed all checks.
    """
    issues: list[str] = []

    item_type = payload.get("type")
    if not isinstance(item_type, str) or item_type not in FHIR_ALLOWED_TYPES:
        issues.append("invalid_type")

    link_id = payload.get("linkId")
    if not isinstance(link_id, str) or not link_id.strip():
        issues.append("invalid_linkId")

    if "required" in payload and not isinstance(payload.get("required"), bool):
        issues.append("invalid_required")

    if item_type not in {"group", "display"}:
        text = payload.get("text")
        if not isinstance(text, str) or not text.strip():
            issues.append("missing_text")

        if not is_valid_code_array(payload.get("code")):
            issues.append("invalid_code")

    if item_type in {"choice", "open-choice"}:
        if not has_valid_answer_option(payload):
            issues.append("invalid_answerOption")

    return issues


def run_fhir_audit(records: list[RecordParseResult]) -> AuditSummary:
    """Audit all parsed records and produce dataset-level statistics.

    Parameters
    ----------
    records:
        Parsed records from `load_fhir_jsonl_records`.

    Returns
    -------
    AuditSummary
        Aggregated validity metrics, type distribution, and issue frequencies.
    """
    type_counter: Counter[str] = Counter()
    issue_counter: Counter[str] = Counter()

    valid_records = 0

    for rec in records:
        payload = rec.assistant_obj
        item_type = payload.get("type")
        if isinstance(item_type, str):
            type_counter[item_type] += 1
        else:
            type_counter["<missing-or-invalid>"] += 1

        issues = audit_fhir_record(payload)
        if len(issues) == 0:
            valid_records += 1
        else:
            issue_counter.update(issues)

    total_records = len(records)
    invalid_records = total_records - valid_records
    pass_rate = (valid_records / total_records * 100.0) if total_records else 0.0

    return AuditSummary(
        total_records=total_records,
        valid_records=valid_records,
        invalid_records=invalid_records,
        pass_rate=pass_rate,
        type_counts=dict(sorted(type_counter.items(), key=lambda x: x[0])),
        issue_counts=dict(sorted(issue_counter.items(), key=lambda x: (-x[1], x[0]))),
    )


audit_summary = run_fhir_audit(parsed_records)

print("FHIR Audit Summary")
print("-" * 60)
print(f"Total records:   {audit_summary.total_records}")
print(f"Valid records:   {audit_summary.valid_records}")
print(f"Invalid records: {audit_summary.invalid_records}")
print(f"Pass rate:       {audit_summary.pass_rate:.2f}%")

print("\nType distribution:")
for t, count in audit_summary.type_counts.items():
    print(f"- {t}: {count}")

print("\nIssue counts:")
if audit_summary.issue_counts:
    for issue, count in audit_summary.issue_counts.items():
        print(f"- {issue}: {count}")
else:
    print("- none")

FHIR Audit Summary
------------------------------------------------------------
Total records:   150
Valid records:   150
Invalid records: 0
Pass rate:       100.00%

Type distribution:
- boolean: 8
- choice: 52
- date: 8
- dateTime: 6
- decimal: 10
- display: 5
- group: 5
- integer: 17
- open-choice: 11
- string: 8
- text: 8
- time: 6
- url: 6

Issue counts:
- none


## Training Configuration and Rationale

### Objective

We fine-tune **mlx-community/Qwen2.5-7B-Instruct-4bit** to generate **FHIR R4 QuestionnaireItem-like JSON** from ChatML prompts, prioritizing:

- structural validity,
- schema conformance,
- robust behavior across all 13 item types.

### Adaptation Strategy

We use a parameter-efficient setup (LoRA/QLoRA-style) to reduce memory footprint and improve practical reproducibility on Apple Silicon.

Why this choice:

- lower compute requirements than full fine-tuning,
- faster iteration cycles for ablation and error analysis,
- easier artifact sharing (adapter weights instead of full model checkpoints).

### Core Hyperparameter Dimensions

The experiment tracks and justifies:

1. model identifier and tokenizer pairing,
2. LoRA rank and scaling parameters,
3. sequence length and effective batch size,
4. optimizer and learning rate schedule,
5. number of training steps / iterations,
6. validation interval and checkpoint cadence.

### Reproducibility Controls

This training section is tied to:

- fixed global and split seeds,
- explicit train/validation manifest,
- persisted runtime metadata,
- versioned configuration snapshot.

### Reporting Policy

For academic transparency, we will report:

- final hyperparameter configuration,
- training dynamics (loss curve, runtime),
- validation metrics and FHIR pass-rate,
- representative qualitative generations.

### References

- LoRA: https://arxiv.org/abs/2106.09685
- QLoRA: https://arxiv.org/abs/2305.14314
- MLX-LM: https://github.com/ml-explore/mlx-lm

In [4]:
# Reproducible train/validation split (stratified by FHIR item type)

from __future__ import annotations

import json
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

SPLIT_SEED = 173
VAL_RATIO = 0.20


def get_item_type(payload: dict[str, Any]) -> str:
    """Return normalized item type for split stratification.

    Parameters
    ----------
    payload:
        Assistant JSON object for one dataset record.

    Returns
    -------
    str
        Item type if present and string; otherwise '<missing-or-invalid>'.
    """
    item_type = payload.get("type")
    return item_type if isinstance(item_type, str) and item_type.strip() else "<missing-or-invalid>"


def build_stratified_buckets(records: list[RecordParseResult]) -> dict[str, list[int]]:
    """Group record indices by FHIR item type.

    Parameters
    ----------
    records:
        Parsed dataset records.

    Returns
    -------
    dict[str, list[int]]
        Mapping from item type to list of record indices.
    """
    buckets: dict[str, list[int]] = defaultdict(list)
    for idx, rec in enumerate(records):
        buckets[get_item_type(rec.assistant_obj)].append(idx)
    return dict(buckets)


def stratified_train_val_split(
    records: list[RecordParseResult],
    val_ratio: float,
    seed: int,
) -> tuple[list[int], list[int]]:
    """Create deterministic train/validation indices with type stratification.

    Parameters
    ----------
    records:
        Parsed dataset records.
    val_ratio:
        Fraction of data assigned to validation set.
    seed:
        Random seed used for deterministic shuffling.

    Returns
    -------
    tuple[list[int], list[int]]
        Train indices and validation indices.
    """
    if not (0.0 < val_ratio < 1.0):
        raise ValueError("val_ratio must be between 0 and 1")
    if len(records) < 2:
        raise ValueError("At least two records are required for splitting")

    rng = random.Random(seed)
    buckets = build_stratified_buckets(records)

    train_idx: list[int] = []
    val_idx: list[int] = []

    for _, indices in buckets.items():
        shuffled = indices[:]
        rng.shuffle(shuffled)

        n_total = len(shuffled)
        n_val = max(1, int(round(n_total * val_ratio))) if n_total > 1 else 0
        n_val = min(n_val, n_total - 1) if n_total > 1 else 0

        val_idx.extend(shuffled[:n_val])
        train_idx.extend(shuffled[n_val:])

    train_idx.sort()
    val_idx.sort()

    if set(train_idx).intersection(val_idx):
        raise RuntimeError("Train/validation overlap detected")
    if len(train_idx) + len(val_idx) != len(records):
        raise RuntimeError("Split size mismatch")

    return train_idx, val_idx


def summarize_type_distribution(records: list[RecordParseResult], indices: list[int]) -> dict[str, int]:
    """Summarize type counts for a subset of records.

    Parameters
    ----------
    records:
        Parsed dataset records.
    indices:
        Selected subset indices.

    Returns
    -------
    dict[str, int]
        Sorted count dictionary by item type.
    """
    counter: Counter[str] = Counter()
    for idx in indices:
        counter[get_item_type(records[idx].assistant_obj)] += 1
    return dict(sorted(counter.items(), key=lambda x: x[0]))


def save_split_manifest(
    output_path: Path,
    train_idx: list[int],
    val_idx: list[int],
    seed: int,
    val_ratio: float,
    train_dist: dict[str, int],
    val_dist: dict[str, int],
) -> None:
    """Persist split metadata for reproducibility and auditability.

    Parameters
    ----------
    output_path:
        JSON file path where split metadata will be written.
    train_idx:
        Train subset indices.
    val_idx:
        Validation subset indices.
    seed:
        Seed used to generate the split.
    val_ratio:
        Validation ratio used during splitting.
    train_dist:
        Type distribution for train subset.
    val_dist:
        Type distribution for validation subset.
    """
    payload = {
        "seed": seed,
        "val_ratio": val_ratio,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "train_indices": train_idx,
        "val_indices": val_idx,
        "train_type_distribution": train_dist,
        "val_type_distribution": val_dist,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


train_indices, val_indices = stratified_train_val_split(
    records=parsed_records,
    val_ratio=VAL_RATIO,
    seed=SPLIT_SEED,
)

train_distribution = summarize_type_distribution(parsed_records, train_indices)
val_distribution = summarize_type_distribution(parsed_records, val_indices)

split_manifest_path = repo_root / "lab" / "artifacts" / "split_manifest.json"
save_split_manifest(
    output_path=split_manifest_path,
    train_idx=train_indices,
    val_idx=val_indices,
    seed=SPLIT_SEED,
    val_ratio=VAL_RATIO,
    train_dist=train_distribution,
    val_dist=val_distribution,
)

print("Split summary")
print("-" * 60)
print(f"Total records: {len(parsed_records)}")
print(f"Train size:    {len(train_indices)}")
print(f"Val size:      {len(val_indices)}")
print(f"Seed:          {SPLIT_SEED}")
print(f"Manifest:      {split_manifest_path}")

print("\nTrain type distribution:")
for t, c in train_distribution.items():
    print(f"- {t}: {c}")

print("\nValidation type distribution:")
for t, c in val_distribution.items():
    print(f"- {t}: {c}")

Split summary
------------------------------------------------------------
Total records: 150
Train size:    120
Val size:      30
Seed:          173
Manifest:      /Users/CAE9/aiprom-llm/lab/artifacts/split_manifest.json

Train type distribution:
- boolean: 6
- choice: 42
- date: 6
- dateTime: 5
- decimal: 8
- display: 4
- group: 4
- integer: 14
- open-choice: 9
- string: 6
- text: 6
- time: 5
- url: 5

Validation type distribution:
- boolean: 2
- choice: 10
- date: 2
- dateTime: 1
- decimal: 2
- display: 1
- group: 1
- integer: 3
- open-choice: 2
- string: 2
- text: 2
- time: 1
- url: 1


In [5]:
# Training configuration: canonical object + persistence for reproducibility

from __future__ import annotations

import json
from datetime import datetime, UTC
from pathlib import Path
from typing import Any


def build_training_config(
    model_name: str,
    train_size: int,
    val_size: int,
    seed: int,
) -> dict[str, Any]:
    """Build a reproducible training configuration dictionary.

    Parameters
    ----------
    model_name:
        Base model identifier used for fine-tuning.
    train_size:
        Number of training examples.
    val_size:
        Number of validation examples.
    seed:
        Global random seed used across the notebook.

    Returns
    -------
    dict[str, Any]
        Canonical configuration payload including model, optimization,
        data, and logging settings.
    """
    return {
        "experiment_name": "qwen_public_fhir_lora_v1",
        "created_utc": datetime.now(UTC).isoformat(),
        "model": {
            "name": model_name,
            "task": "chatml_to_fhir_questionnaireitem_json",
        },
        "data": {
            "dataset_path": str(dataset_path),
            "n_train": train_size,
            "n_val": val_size,
            "split_manifest_path": str(repo_root / "lab" / "artifacts" / "split_manifest.json"),
        },
        "reproducibility": {
            "global_seed": seed,
            "split_seed": seed,
            "python_version": runtime_info.get("python_version"),
            "platform": runtime_info.get("platform"),
            "git_commit": runtime_info.get("git_commit"),
        },
        "training": {
            "strategy": "lora_or_qlora",
            "max_seq_len": 2048,
            "batch_size": 2,
            "grad_accum_steps": 8,
            "learning_rate": 2e-4,
            "weight_decay": 0.0,
            "warmup_steps": 20,
            "train_steps": 400,
            "eval_interval": 25,
            "save_interval": 100,
        },
        "lora": {
            "rank": 16,
            "alpha": 32,
            "dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        },
        "outputs": {
            "artifact_dir": str(repo_root / "lab" / "artifacts"),
            "checkpoint_dir": str(repo_root / "lab" / "artifacts" / "checkpoints"),
            "log_json_path": str(repo_root / "lab" / "artifacts" / "training_config.json"),
        },
    }


def persist_training_config(config: dict[str, Any], output_path: Path) -> None:
    """Write training configuration to disk in JSON format.

    Parameters
    ----------
    config:
        Training configuration dictionary.
    output_path:
        Destination JSON path for persistence.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(config, indent=2), encoding="utf-8")


def print_training_config_summary(config: dict[str, Any]) -> None:
    """Print a compact summary of the current training setup.

    Parameters
    ----------
    config:
        Training configuration dictionary.
    """
    print("Training configuration summary")
    print("-" * 60)
    print(f"Experiment:     {config['experiment_name']}")
    print(f"Model:          {config['model']['name']}")
    print(f"Task:           {config['model']['task']}")
    print(f"Train/Val:      {config['data']['n_train']} / {config['data']['n_val']}")
    print(f"Seed:           {config['reproducibility']['global_seed']}")
    print(f"Train steps:    {config['training']['train_steps']}")
    print(f"Learning rate:  {config['training']['learning_rate']}")
    print(f"Batch size:     {config['training']['batch_size']}")
    print(f"Grad accum:     {config['training']['grad_accum_steps']}")
    print(f"Seq length:     {config['training']['max_seq_len']}")
    print(f"Config JSON:    {config['outputs']['log_json_path']}")


MODEL_NAME = "mlx-community/Qwen2.5-7B-Instruct-4bit"

training_config = build_training_config(
    model_name=MODEL_NAME,
    train_size=len(train_indices),
    val_size=len(val_indices),
    seed=GLOBAL_SEED,
 )

config_path = Path(training_config["outputs"]["log_json_path"] )
persist_training_config(training_config, config_path)
print_training_config_summary(training_config)

Training configuration summary
------------------------------------------------------------
Experiment:     qwen_public_fhir_lora_v1
Model:          mlx-community/Qwen2.5-7B-Instruct-4bit
Task:           chatml_to_fhir_questionnaireitem_json
Train/Val:      120 / 30
Seed:           173
Train steps:    400
Learning rate:  0.0002
Batch size:     2
Grad accum:     8
Seq length:     2048
Config JSON:    /Users/CAE9/aiprom-llm/lab/artifacts/training_config.json


## Data Materialization for Training

This section converts parsed records and deterministic split indices into train/validation files ready for MLX-LM LoRA training.

### Goals

- Produce reproducible train/validation JSONL artifacts.
- Preserve original ChatML transcripts.
- Persist metadata for traceability and later audit.

### Output Artifacts

- `lab/artifacts/train.jsonl`
- `lab/artifacts/val.jsonl`
- `lab/artifacts/dataset_manifest.json`

These files are generated deterministically from the parsed dataset and split manifest.

In [6]:
# Build deterministic train/validation files for MLX-LM

from __future__ import annotations

import json
from pathlib import Path
from typing import Any


def build_chatml_record(text: str) -> dict[str, str]:
    """Create a minimal MLX-LM-compatible JSONL record from ChatML text.

    Parameters
    ----------
    text:
        ChatML transcript string containing system/user/assistant turns.

    Returns
    -------
    dict[str, str]
        Single record with one `text` field.
    """
    return {"text": text}


def materialize_split_jsonl(
    records: list[RecordParseResult],
    indices: list[int],
    output_path: Path,
) -> None:
    """Write selected records into JSONL format.

    Parameters
    ----------
    records:
        Parsed dataset records.
    indices:
        Record indices to include in the output file.
    output_path:
        Destination JSONL path.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for idx in indices:
            row = build_chatml_record(records[idx].text)
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_dataset_manifest(
    manifest_path: Path,
    train_path: Path,
    val_path: Path,
    valid_path: Path,
    train_size: int,
    val_size: int,
) -> None:
    """Persist dataset materialization metadata for reproducibility.

    Parameters
    ----------
    manifest_path:
        Output path for JSON manifest.
    train_path:
        Train JSONL artifact path.
    val_path:
        Validation JSONL artifact path.
    valid_path:
        Validation JSONL alias path for tool compatibility.
    train_size:
        Number of train records.
    val_size:
        Number of validation records.
    """
    payload = {
        "dataset_source": str(dataset_path),
        "train_path": str(train_path),
        "val_path": str(val_path),
        "valid_path": str(valid_path),
        "n_train": train_size,
        "n_val": val_size,
        "split_seed": SPLIT_SEED,
        "global_seed": GLOBAL_SEED,
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


artifacts_dir = repo_root / "lab" / "artifacts"
train_jsonl_path = artifacts_dir / "train.jsonl"
val_jsonl_path = artifacts_dir / "val.jsonl"
valid_jsonl_path = artifacts_dir / "valid.jsonl"
dataset_manifest_path = artifacts_dir / "dataset_manifest.json"

materialize_split_jsonl(parsed_records, train_indices, train_jsonl_path)
materialize_split_jsonl(parsed_records, val_indices, val_jsonl_path)
materialize_split_jsonl(parsed_records, val_indices, valid_jsonl_path)
write_dataset_manifest(
    dataset_manifest_path,
    train_jsonl_path,
    val_jsonl_path,
    valid_jsonl_path,
    len(train_indices),
    len(val_indices),
)

print("Materialization complete")
print("-" * 60)
print(f"Train JSONL: {train_jsonl_path}")
print(f"Val JSONL:   {val_jsonl_path}")
print(f"Valid JSONL: {valid_jsonl_path}")
print(f"Manifest:    {dataset_manifest_path}")
print(f"Train rows:  {len(train_indices)}")
print(f"Val rows:    {len(val_indices)}")

Materialization complete
------------------------------------------------------------
Train JSONL: /Users/CAE9/aiprom-llm/lab/artifacts/train.jsonl
Val JSONL:   /Users/CAE9/aiprom-llm/lab/artifacts/val.jsonl
Valid JSONL: /Users/CAE9/aiprom-llm/lab/artifacts/valid.jsonl
Manifest:    /Users/CAE9/aiprom-llm/lab/artifacts/dataset_manifest.json
Train rows:  120
Val rows:    30


## Training Execution (MLX-LM)

This section launches LoRA fine-tuning using MLX-LM with the persisted configuration.

### Notes

- The command is generated programmatically from `training_config`.
- If the training command fails due to environment-specific CLI differences, the notebook reports stderr and keeps reproducibility artifacts.
- Output checkpoints are written under `lab/artifacts/checkpoints`.

In [7]:
# Construct and execute MLX-LM training command

from __future__ import annotations

import json
import shlex
import subprocess
from pathlib import Path
from typing import Any

RUN_TRAINING = True

def get_mlx_lm_lora_help(cwd: Path) -> str:
    """Return help text for `python -m mlx_lm lora`."""
    completed = subprocess.run(
        ["python", "-m", "mlx_lm", "lora", "--help"],
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    return (completed.stdout or "") + "\n" + (completed.stderr or "")

def build_mlx_lm_lora_command(config: dict[str, Any], help_text: str) -> list[str]:
    """Build MLX-LM LoRA command line arguments from config."""
    training = config["training"]
    outputs = config["outputs"]
    model_name = str(config["model"]["name"]).strip()

    if not model_name.startswith("mlx-community/"):
        raise ValueError(
            f"Invalid base model for this notebook: {model_name}. "
            "Use a public mlx-community model to avoid auth failures."
        )

    command = [
        "python",
        "-m",
        "mlx_lm",
        "lora",
        "--model",
        model_name,
        "--train",
        "--data",
        str(artifacts_dir),
        "--iters",
        str(training["train_steps"]),
        "--batch-size",
        str(training["batch_size"]),
        "--learning-rate",
        str(training["learning_rate"]),
        "--steps-per-report",
        str(training["eval_interval"]),
        "--steps-per-eval",
        str(training["eval_interval"]),
        "--save-every",
        str(training["save_interval"]),
        "--adapter-path",
        str(Path(outputs["checkpoint_dir"])),
    ]

    if "--grad-accumulation-steps" in help_text:
        command.extend(["--grad-accumulation-steps", str(training["grad_accum_steps"])])
    if "--max-seq-length" in help_text:
        command.extend(["--max-seq-length", str(training["max_seq_len"])])
    if "--seed" in help_text:
        command.extend(["--seed", str(config["reproducibility"]["global_seed"])])

    return command

def run_command(command: list[str], cwd: Path) -> tuple[int, str, str]:
    """Run a subprocess command and capture stdout/stderr."""
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    return completed.returncode, completed.stdout, completed.stderr

def persist_run_log(log_path: Path, payload: dict[str, Any]) -> None:
    """Write training run log payload to JSON."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def adapter_artifacts_exist(checkpoint_dir: Path) -> bool:
    """Return True if adapter files exist in checkpoint directory tree."""
    markers = {"adapters.safetensors", "adapter_model.safetensors"}
    if not checkpoint_dir.exists():
        return False
    for file in checkpoint_dir.rglob("*"):
        if file.is_file() and file.name in markers:
            return True
    return False

help_text = get_mlx_lm_lora_help(repo_root)
train_command = build_mlx_lm_lora_command(training_config, help_text)
train_command_str = " ".join(shlex.quote(x) for x in train_command)
checkpoint_dir = Path(training_config["outputs"]["checkpoint_dir"])

print("Training command")
print("-" * 60)
print(train_command_str)

if not RUN_TRAINING:
    print("\nRUN_TRAINING is False: skipping execution (command validated only).")
else:
    return_code, stdout, stderr = run_command(train_command, repo_root)

    training_run_log = {
        "command": train_command,
        "return_code": return_code,
        "stdout_preview": stdout[-4000:],
        "stderr_preview": stderr[-4000:],
    }

    training_run_log_path = artifacts_dir / "training_run_log.json"
    persist_run_log(training_run_log_path, training_run_log)

    print("\nTraining execution result")
    print("-" * 60)
    print(f"Return code: {return_code}")
    print(f"Run log:     {training_run_log_path}")
    print(f"Checkpoint:  {checkpoint_dir}")

    if return_code != 0:
        print("\nTraining command returned non-zero exit code.")
        print("Last stderr lines:")
        print(stderr[-1200:])
        raise RuntimeError("LoRA training failed; adapter was not generated.")

    if not adapter_artifacts_exist(checkpoint_dir):
        raise RuntimeError(
            "Training finished but no adapter artifacts were found. "
            "Expected adapters.safetensors/adapter_model.safetensors under checkpoint_dir."
        )

    print("Training finished successfully and adapter artifacts were detected.")

Training command
------------------------------------------------------------
python -m mlx_lm lora --model mlx-community/Qwen2.5-7B-Instruct-4bit --train --data /Users/CAE9/aiprom-llm/lab/artifacts --iters 400 --batch-size 2 --learning-rate 0.0002 --steps-per-report 25 --steps-per-eval 25 --save-every 100 --adapter-path /Users/CAE9/aiprom-llm/lab/artifacts/checkpoints --grad-accumulation-steps 8 --max-seq-length 2048 --seed 173

Training execution result
------------------------------------------------------------
Return code: 0
Run log:     /Users/CAE9/aiprom-llm/lab/artifacts/training_run_log.json
Checkpoint:  /Users/CAE9/aiprom-llm/lab/artifacts/checkpoints
Training finished successfully and adapter artifacts were detected.


## GGUF Deliverable for LM Studio

This section fuses the trained LoRA adapter into the base model and exports a GGUF artifact for LM Studio.

### Purpose

- Produce a single portable GGUF model file.
- Avoid loading LoRA separately inside LM Studio.
- Keep export metadata reproducible in `lab/artifacts/`.

Set `RUN_GGUF_EXPORT = True` to execute the export command.

In [10]:
# Fuse LoRA adapter and export GGUF for LM Studio

from __future__ import annotations

import json
import shlex
import subprocess
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

RUN_GGUF_EXPORT = False

ROOT_DIR = Path(repo_root) if "repo_root" in globals() else Path.cwd()
ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (ROOT_DIR / "lab" / "artifacts")
BASE_MODEL_NAME = str(training_config.get("model", {}).get("name", "mlx-community/Qwen2.5-7B-Instruct-4bit")) if "training_config" in globals() else "mlx-community/Qwen2.5-7B-Instruct-4bit"

def candidate_adapter_dirs(base_dir: Path) -> list[Path]:
    """Return likely adapter/checkpoint directories generated by MLX-LM LoRA training."""
    candidates = [
        base_dir / "adapters",
        base_dir / "checkpoints",
        base_dir / "checkpoint",
    ]
    return [p for p in candidates if p.exists() and p.is_dir()]

def select_adapter_path(base_dir: Path) -> Path | None:
    """Select adapter directory if any candidate contains adapter artifacts."""
    for path in candidate_adapter_dirs(base_dir):
        for marker in ("adapters.safetensors", "adapter_model.safetensors"):
            if (path / marker).exists():
                return path
    return None

def build_fuse_export_command(
    *,
    model_name: str,
    adapter_path: Path,
    fused_dir: Path,
    gguf_path: Path,
 ) -> list[str]:
    """Build MLX-LM command for fuse + GGUF export."""
    return [
        "python",
        "-m",
        "mlx_lm",
        "fuse",
        "--model",
        model_name,
        "--adapter-path",
        str(adapter_path),
        "--save-path",
        str(fused_dir),
        "--export-gguf",
        "--gguf-path",
        str(gguf_path),
    ]

def run_command(command: list[str], cwd: Path) -> tuple[int, str, str]:
    """Run command and return returncode/stdout/stderr."""
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    return completed.returncode, completed.stdout, completed.stderr

def save_gguf_export_log(log_path: Path, payload: dict[str, Any]) -> None:
    """Persist GGUF export run metadata."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

adapter_path = select_adapter_path(ARTIFACTS_DIR)
fused_dir = ARTIFACTS_DIR / "fused"
gguf_path = ARTIFACTS_DIR / "fused" / "model-fused.gguf"
log_path = ARTIFACTS_DIR / "gguf_export_log.json"

print("GGUF export plan")
print("-" * 60)
print(f"Base model:     {BASE_MODEL_NAME}")
print(f"Adapter path:   {adapter_path if adapter_path else '<not-found>'}")
print(f"Fused dir:      {fused_dir}")
print(f"GGUF path:      {gguf_path}")
print(f"Run enabled:    {RUN_GGUF_EXPORT}")

if not adapter_path:
    print("\nNo adapter artifacts found. Run training first, then re-run this cell.")
elif not RUN_GGUF_EXPORT:
    preview_command = build_fuse_export_command(
        model_name=BASE_MODEL_NAME,
        adapter_path=adapter_path,
        fused_dir=fused_dir,
        gguf_path=gguf_path,
    )
    print("\nRUN_GGUF_EXPORT is False. Command preview:")
    print(shlex.join(preview_command))
else:
    fuse_command = build_fuse_export_command(
        model_name=BASE_MODEL_NAME,
        adapter_path=adapter_path,
        fused_dir=fused_dir,
        gguf_path=gguf_path,
    )
    return_code, stdout_text, stderr_text = run_command(fuse_command, ROOT_DIR)

    payload = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "command": fuse_command,
        "base_model": BASE_MODEL_NAME,
        "adapter_path": str(adapter_path),
        "fused_dir": str(fused_dir),
        "gguf_path": str(gguf_path),
        "return_code": return_code,
        "stdout": stdout_text,
        "stderr": stderr_text,
    }
    save_gguf_export_log(log_path, payload)

    print("\nGGUF export result")
    print("-" * 60)
    print(f"Return code: {return_code}")
    print(f"Log file:    {log_path}")
    print(f"GGUF exists: {gguf_path.exists()}")

    if return_code != 0:
        print("\nExport command failed. Last stderr chunk:")
        print(stderr_text[-1500:])
    else:
        print("GGUF deliverable generated successfully.")

payload if "payload" in locals() else None

GGUF export plan
------------------------------------------------------------
Base model:     mlx-community/Qwen2.5-7B-Instruct-4bit
Adapter path:   /Users/CAE9/aiprom-llm/lab/artifacts/checkpoints
Fused dir:      /Users/CAE9/aiprom-llm/lab/artifacts/fused
GGUF path:      /Users/CAE9/aiprom-llm/lab/artifacts/fused/model-fused.gguf
Run enabled:    False

RUN_GGUF_EXPORT is False. Command preview:
python -m mlx_lm fuse --model mlx-community/Qwen2.5-7B-Instruct-4bit --adapter-path /Users/CAE9/aiprom-llm/lab/artifacts/checkpoints --save-path /Users/CAE9/aiprom-llm/lab/artifacts/fused --export-gguf --gguf-path /Users/CAE9/aiprom-llm/lab/artifacts/fused/model-fused.gguf


{'timestamp_utc': '2026-02-22T17:03:13.033125+00:00',
 'command': ['python',
  '-m',
  'mlx_lm',
  'fuse',
  '--model',
  'mlx-community/Qwen2.5-7B-Instruct-4bit',
  '--adapter-path',
  '/Users/CAE9/aiprom-llm/lab/artifacts/checkpoints',
  '--save-path',
  '/Users/CAE9/aiprom-llm/lab/artifacts/fused',
  '--export-gguf',
  '--gguf-path',
  '/Users/CAE9/aiprom-llm/lab/artifacts/fused/model-fused.gguf'],
 'base_model': 'mlx-community/Qwen2.5-7B-Instruct-4bit',
 'adapter_path': '/Users/CAE9/aiprom-llm/lab/artifacts/checkpoints',
 'fused_dir': '/Users/CAE9/aiprom-llm/lab/artifacts/fused',
 'gguf_path': '/Users/CAE9/aiprom-llm/lab/artifacts/fused/model-fused.gguf',
 'return_code': 1,
 'stdout': 'Loading pretrained model\n',
 'stderr': '\nFetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]\nFetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 9647.01it/s]\nTraceback (most recent call last):\n  File \x1b"<frozen runpy>"\x1b, line \x1b198\x1b, in \x1b_run_module_as_main\x1b\n  File \x1b"<fr

## Adapter Inference Test

This section runs a quick inference sanity check using the trained LoRA adapter.

### Purpose

- Verify adapter loading works.
- Generate one FHIR-style response from a ChatML prompt.
- Persist a reproducible inference log artifact.

Set `RUN_INFERENCE_TEST = True` to execute generation.

In [11]:
# Quick inference test with trained adapter (strict: no fallback)

from __future__ import annotations

import json
import shlex
import subprocess
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

RUN_INFERENCE_TEST = True

ROOT_DIR = Path(repo_root) if "repo_root" in globals() else Path.cwd()
ARTIFACTS_DIR = Path(artifacts_dir) if "artifacts_dir" in globals() else (ROOT_DIR / "lab" / "artifacts")
DEFAULT_PUBLIC_MODEL = "mlx-community/Qwen2.5-7B-Instruct-4bit"

def build_chatml_test_prompt() -> str:
    """Build a deterministic ChatML prompt for adapter inference sanity check."""
    return (
        "<|im_start|>system\n"
        "You are a FHIR R4 expert. Generate VALID FHIR R4 Questionnaire JSON only.\n"
        "<|im_end|>"
        "<|im_start|>user\n"
        "Generate a complete FHIR Questionnaire for a weekly recovery check-in form. "
        "Include resourceType='Questionnaire', id='weekly-recovery-checkin', status='active', title, and an item array with at least 6 questions. "
        "Use mixed item types (choice, string, text, integer, boolean, date) and valid linkId values. "
        "For choice items include 4 answerOption values with valueCoding.display. Return only JSON.\n"
        "<|im_end|>"
        "<|im_start|>assistant\n"
    )

def resolve_inference_model_name() -> str:
    """Resolve model name robustly, preferring public mlx-community checkpoints."""
    configured = ""
    if "training_config" in globals() and isinstance(training_config, dict):
        configured = str(training_config.get("model", {}).get("name", "")).strip()

    model_from_globals = str(globals().get("MODEL_NAME", "")).strip()

    candidates = [configured, model_from_globals, DEFAULT_PUBLIC_MODEL]
    for candidate in candidates:
        if candidate.startswith("mlx-community/"):
            return candidate

    return DEFAULT_PUBLIC_MODEL

def select_adapter_path(base_dir: Path) -> Path | None:
    """Find adapter directory with supported marker files."""
    candidates = [base_dir / "adapters", base_dir / "checkpoints", base_dir / "checkpoint"]
    for path in candidates:
        if not path.exists() or not path.is_dir():
            continue
        for marker in ("adapters.safetensors", "adapter_model.safetensors"):
            if any(p.name == marker for p in path.rglob("*")):
                return path
    return None

def build_mlx_generate_command(model_name: str, prompt: str, adapter_path: Path) -> list[str]:
    """Build `mlx_lm generate` command with mandatory adapter path."""
    return [
        "python",
        "-m",
        "mlx_lm",
        "generate",
        "--model",
        model_name,
        "--prompt",
        prompt,
        "--max-tokens",
        "900",
        "--adapter-path",
        str(adapter_path),
    ]

def run_inference_command(command: list[str], cwd: Path) -> tuple[int, str, str]:
    """Run inference command and capture outputs."""
    process = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    return process.returncode, process.stdout, process.stderr

def extract_generated_json(stdout: str) -> dict[str, Any] | None:
    """Extract generated JSON object from model stdout when possible."""
    text = stdout.strip()

    fenced_start = text.find("```json")
    if fenced_start >= 0:
        start_idx = fenced_start + len("```json")
        fenced_end = text.find("```", start_idx)
        if fenced_end > start_idx:
            candidate = text[start_idx:fenced_end].strip()
            try:
                parsed = json.loads(candidate)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass

    first_brace = text.find("{")
    last_brace = text.rfind("}")
    if 0 <= first_brace < last_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return None

    return None

def save_inference_log(
    *,
    output_path: Path,
    command: list[str],
    return_code: int,
    stdout: str,
    stderr: str,
    prompt: str,
    model_name: str,
    adapter_path: Path,
    timestamp: str,
    max_tokens: int,
    temperature: float,
    top_p: float,
    generated_json: dict[str, Any] | None,
    notes: str = "",
) -> dict[str, Any]:
    """Save inference execution metadata and outputs to JSON."""
    payload: dict[str, Any] = {
        "timestamp_utc": timestamp,
        "model": model_name,
        "adapter_path": str(adapter_path),
        "prompt": prompt,
        "command": command,
        "generation": {
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p,
        },
        "result": {
            "return_code": return_code,
            "stdout": stdout,
            "stderr": stderr,
            "generated_json": generated_json,
        },
    }
    if notes:
        payload["notes"] = notes

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return payload

if RUN_INFERENCE_TEST:
    model_name = resolve_inference_model_name()
    adapter_path = select_adapter_path(ARTIFACTS_DIR)
    if adapter_path is None:
        raise RuntimeError(
            "Adapter LoRA no encontrado. Este notebook no permite fallback al modelo base. "
            "Ejecuta primero la celda de entrenamiento y verifica artefactos en lab/artifacts/checkpoints."
        )

    prompt = build_chatml_test_prompt()
    command = build_mlx_generate_command(model_name, prompt, adapter_path)
    return_code, stdout_text, stderr_text = run_inference_command(command, ROOT_DIR)
    generated_json = extract_generated_json(stdout_text)

    timestamp = datetime.now(UTC).isoformat()
    log_path = ARTIFACTS_DIR / "inference_test_log.json"
    payload = save_inference_log(
        output_path=log_path,
        command=command,
        return_code=return_code,
        stdout=stdout_text,
        stderr=stderr_text,
        prompt=prompt,
        model_name=model_name,
        adapter_path=adapter_path,
        timestamp=timestamp,
        max_tokens=900,
        temperature=0.0,
        top_p=1.0,
        generated_json=generated_json,
        notes="Inference sanity check on complete FHIR Questionnaire generation using trained adapter.",
    )

    print("Inference mode: adapter-only")
    print(f"Model used: {model_name}")
    print(f"Adapter path: {adapter_path}")
    print(f"Inference command: {shlex.join(command)}")
    print(f"Return code: {return_code}")
    print(f"Inference log saved to: {log_path}")
    if stderr_text.strip():
        print("\n--- STDERR (truncated to 1200 chars) ---")
        print(stderr_text[:1200])

    if generated_json is not None:
        print("\n--- GENERATED JSON (pretty) ---")
        print(json.dumps(generated_json, ensure_ascii=False, indent=2))
    else:
        print("\nNo se pudo extraer un JSON generado de STDOUT.")

    if return_code != 0:
        raise RuntimeError("Inference command failed. Inspect stderr/log for details.")

    payload

Inference mode: adapter-only
Model used: mlx-community/Qwen2.5-7B-Instruct-4bit
Adapter path: /Users/CAE9/aiprom-llm/lab/artifacts/checkpoints
Inference command: python -m mlx_lm generate --model mlx-community/Qwen2.5-7B-Instruct-4bit --prompt '<|im_start|>system
You are a FHIR R4 expert. Generate VALID FHIR R4 Questionnaire JSON only.
<|im_end|><|im_start|>user
Generate a complete FHIR Questionnaire for a weekly recovery check-in form. Include resourceType='"'"'Questionnaire'"'"', id='"'"'weekly-recovery-checkin'"'"', status='"'"'active'"'"', title, and an item array with at least 6 questions. Use mixed item types (choice, string, text, integer, boolean, date) and valid linkId values. For choice items include 4 answerOption values with valueCoding.display. Return only JSON.
<|im_end|><|im_start|>assistant
' --max-tokens 900 --adapter-path /Users/CAE9/aiprom-llm/lab/artifacts/checkpoints
Return code: 0
Inference log saved to: /Users/CAE9/aiprom-llm/lab/artifacts/inference_test_log.json

## Quantitative Evaluation

This section computes deterministic post-training quality indicators, including:

- dataset-level JSON validity,
- FHIR structural pass-rate,
- type-wise support and conformance,
- summary tables for reporting.

When no model generations are available yet, this section still reports gold-dataset structural metrics as a baseline.

In [ ]:
# Compute quantitative summary tables

from __future__ import annotations

from collections import Counter
from typing import Any


def build_type_quality_table(records: list[RecordParseResult]) -> list[dict[str, Any]]:
    """Build per-type structural quality summary using audit rules.

    Parameters
    ----------
    records:
        Parsed dataset records.

    Returns
    -------
    list[dict[str, Any]]
        Table rows with per-type total, pass, fail, and pass-rate.
    """
    totals: Counter[str] = Counter()
    passed: Counter[str] = Counter()

    for rec in records:
        item_type = rec.assistant_obj.get("type")
        key = item_type if isinstance(item_type, str) else "<missing-or-invalid>"
        totals[key] += 1
        if not audit_fhir_record(rec.assistant_obj):
            passed[key] += 1

    rows: list[dict[str, Any]] = []
    for t in sorted(totals.keys()):
        n_total = totals[t]
        n_pass = passed[t]
        n_fail = n_total - n_pass
        pass_rate = (n_pass / n_total * 100.0) if n_total else 0.0
        rows.append(
            {
                "type": t,
                "total": n_total,
                "pass": n_pass,
                "fail": n_fail,
                "pass_rate_percent": round(pass_rate, 2),
            }
        )
    return rows


def pretty_print_table(rows: list[dict[str, Any]], title: str) -> None:
    """Print a compact plain-text table.

    Parameters
    ----------
    rows:
        Row dictionaries with homogeneous keys.
    title:
        Table title to display before rows.
    """
    print(title)
    print("-" * 60)
    if not rows:
        print("<empty>")
        return

    headers = list(rows[0].keys())
    widths = {h: max(len(h), max(len(str(r[h])) for r in rows)) for h in headers}

    header_line = " | ".join(h.ljust(widths[h]) for h in headers)
    sep_line = "-+-".join("-" * widths[h] for h in headers)
    print(header_line)
    print(sep_line)

    for r in rows:
        print(" | ".join(str(r[h]).ljust(widths[h]) for h in headers))


quality_rows = build_type_quality_table(parsed_records)
pretty_print_table(quality_rows, "FHIR structural quality by item type")

overall_pass = sum(r["pass"] for r in quality_rows)
overall_total = sum(r["total"] for r in quality_rows)
overall_rate = (overall_pass / overall_total * 100.0) if overall_total else 0.0

print("\nOverall structural pass rate")
print("-" * 60)
print(f"Pass: {overall_pass}/{overall_total} ({overall_rate:.2f}%)")

FHIR structural quality by item type
------------------------------------------------------------
type        | total | pass | fail | pass_rate_percent
------------+-------+------+------+------------------
boolean     | 8     | 8    | 0    | 100.0            
choice      | 52    | 52   | 0    | 100.0            
date        | 8     | 8    | 0    | 100.0            
dateTime    | 6     | 6    | 0    | 100.0            
decimal     | 10    | 10   | 0    | 100.0            
display     | 5     | 5    | 0    | 100.0            
group       | 5     | 5    | 0    | 100.0            
integer     | 17    | 17   | 0    | 100.0            
open-choice | 11    | 11   | 0    | 100.0            
string      | 8     | 8    | 0    | 100.0            
text        | 8     | 8    | 0    | 100.0            
time        | 6     | 6    | 0    | 100.0            
url         | 6     | 6    | 0    | 100.0            

Overall structural pass rate
------------------------------------------------------------
P

## Qualitative Analysis

This section inspects representative records to support narrative error analysis and reporting.

### Goals

- highlight examples across different FHIR item types,
- inspect structural variety,
- prepare templates for future model-output comparison.

In a later training run, this section can be extended to compare gold references versus generated predictions side by side.

In [ ]:
# Sample qualitative examples by FHIR item type

from __future__ import annotations

import json
import random
from collections import defaultdict
from typing import Any


def sample_examples_by_type(
    records: list[RecordParseResult],
    samples_per_type: int,
    seed: int,
) -> dict[str, list[dict[str, Any]]]:
    """Collect deterministic random samples for each item type.

    Parameters
    ----------
    records:
        Parsed dataset records.
    samples_per_type:
        Number of examples to sample per type.
    seed:
        Random seed for reproducible sampling.

    Returns
    -------
    dict[str, list[dict[str, Any]]]
        Mapping from item type to sampled assistant payloads.
    """
    buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for rec in records:
        item_type = rec.assistant_obj.get("type")
        key = item_type if isinstance(item_type, str) else "<missing-or-invalid>"
        buckets[key].append(rec.assistant_obj)

    rng = random.Random(seed)
    sampled: dict[str, list[dict[str, Any]]] = {}
    for key in sorted(buckets.keys()):
        choices = buckets[key][:]
        rng.shuffle(choices)
        sampled[key] = choices[:samples_per_type]

    return sampled


def print_qualitative_samples(samples: dict[str, list[dict[str, Any]]]) -> None:
    """Pretty-print sampled qualitative records grouped by type.

    Parameters
    ----------
    samples:
        Mapping produced by `sample_examples_by_type`.
    """
    for item_type, rows in samples.items():
        print(f"\nType: {item_type}")
        print("-" * 60)
        if not rows:
            print("<no samples>")
            continue
        for idx, row in enumerate(rows, start=1):
            preview = {
                "linkId": row.get("linkId"),
                "text": row.get("text"),
                "type": row.get("type"),
                "required": row.get("required"),
            }
            print(f"Sample {idx}: {json.dumps(preview, ensure_ascii=False)}")


qual_samples = sample_examples_by_type(
    records=parsed_records,
    samples_per_type=2,
    seed=GLOBAL_SEED,
)
print_qualitative_samples(qual_samples)


Type: boolean
------------------------------------------------------------
Sample 1: {"linkId": "has-hypertension", "text": "Have you been diagnosed with high blood pressure?", "type": "boolean", "required": true}
Sample 2: {"linkId": "current-smoker", "text": "Do you currently smoke?", "type": "boolean", "required": true}

Type: choice
------------------------------------------------------------
Sample 1: {"linkId": "eq5d-self-care", "text": "EQ-5D-5L: Self-Care", "type": "choice", "required": true}
Sample 2: {"linkId": "hads-d1-enjoy", "text": "HADS-D: I still enjoy things I used to", "type": "choice", "required": true}

Type: date
------------------------------------------------------------
Sample 1: {"linkId": "surgery-date", "text": "Date of surgery", "type": "date", "required": false}
Sample 2: {"linkId": "chemo-start-date", "text": "Chemotherapy start date", "type": "date", "required": false}

Type: dateTime
------------------------------------------------------------
Sample 1:

## Limitations, Ethics, and Threats to Validity

### Methodological Limitations

- The dataset size is limited (150 records), increasing overfitting risk.
- Structural validity does not guarantee semantic clinical correctness.
- Prompt distribution may not represent all real-world questionnaire authoring styles.

### External Validity

- Results obtained on this dataset may not transfer to other institutions, languages, or proprietary form libraries without adaptation.

### Ethical and Legal Considerations

- Some PROM/PREM instruments have third-party licensing constraints.
- This notebook is intended for research/educational usage; production or redistribution may require additional permissions.
- Generated clinical forms should be reviewed by qualified professionals before operational use.

### Reproducibility Caveat

- Exact deterministic behavior can vary across platform versions and backend implementations, despite fixed seeds and documented configs.

## References

### Standards and Interoperability

- HL7 FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- FAIR Principles: https://www.go-fair.org/fair-principles/

### Model Adaptation

- LoRA (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- QLoRA (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314

### Tooling

- MLX: https://github.com/ml-explore/mlx
- MLX-LM: https://github.com/ml-explore/mlx-lm
- Qwen docs: https://qwen.readthedocs.io/

### Clinical Instrument Context

- PROMIS (HealthMeasures): https://www.healthmeasures.net/explore-measurement-systems/promis
- EORTC terms and conditions: https://qol.eortc.org/terms-conditions/academic-user/
- EQ-5D access and registration: https://euroqol.org/register/obtain-eq-5d/how-to-obtain-eq-5d/

---

End of notebook.

## How to Cite

If you use this notebook or derived artifacts in academic work, please cite the repository and the methodological references below.

### Suggested citation (repository)

```text
Pimàs, P. (2026). aiprom-llm: Fine-tuning Qwen2.5-7B-Instruct-4bit for FHIR R4 QuestionnaireItem generation [Computer software]. GitHub.
```

### Suggested citation (methods)

- Hu, E. J., et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models. https://arxiv.org/abs/2106.09685
- Dettmers, T., et al. (2023). QLoRA: Efficient Finetuning of Quantized LLMs. https://arxiv.org/abs/2305.14314
- HL7 FHIR R4 Questionnaire specification. https://hl7.org/fhir/R4/questionnaire.html

### Reproducibility citation note

For reproducibility claims, include:

- commit hash,
- training configuration file,
- split manifest,
- dataset manifest,
- runtime platform metadata.

These artifacts are generated in `lab/artifacts/` by this notebook workflow.